# English–Vietnamese Speech API — Kaggle T4x2 Production Showcase

## Before you run / Trước khi chạy

### English

This notebook is the public Kaggle production-showcase template for the qualified **Tesla T4 x2** runtime. Before selecting **Run All**:

1. Create or import the notebook in Kaggle and select **Accelerator → GPU T4 x2**.
2. Keep **Internet = On** so the notebook can clone the public repository and, if explicitly enabled, start a Cloudflare Quick Tunnel.
3. In the **Input** panel, choose **Add Input → Models** and attach the exact three model resources below.

| Model | Kaggle resource | Framework | Variation | Version | Expected read-only mount |
| --- | --- | --- | --- | --- | --- |
| TranslateGemma 4B IT | `keras/translategemma` | Keras | `translategemma_4b_it` | `1` | `/kaggle/input/models/keras/translategemma/keras/translategemma_4b_it/1` |
| Whisper Large V3 Turbo | `dangkhoa2016/openai-whisper-large-v3-turbo` | Transformers | `default` | `1` | `/kaggle/input/models/dangkhoa2016/openai-whisper-large-v3-turbo/transformers/default/1` |
| VoxCPM2 | `dangkhoa2016/openbmb-voxcpm2` | PyTorch | `default` | `1` | `/kaggle/input/models/dangkhoa2016/openbmb-voxcpm2/pytorch/default/1` |

Kaggle mounts attached inputs under `/kaggle/input` as **read-only** storage. Do not attempt to modify model files there. Repository source, virtual environments, logs, generated WAV/ZIP files, benchmark output, and runtime state belong under writable `/kaggle/working`.

Confirm all three model inputs are visible in the Kaggle Input panel before running the notebook.

### Tiếng Việt

Notebook này là template production showcase công khai dành cho runtime **Tesla T4 x2** đã được qualification trên Kaggle. Trước khi chọn **Run All**:

1. Tạo hoặc import notebook vào Kaggle và chọn **Accelerator → GPU T4 x2**.
2. Giữ **Internet = On** để notebook có thể clone repository công khai và, nếu được bật rõ ràng, khởi động Cloudflare Quick Tunnel.
3. Trong panel **Input**, chọn **Add Input → Models** và attach chính xác ba model bên dưới.

| Model | Kaggle resource | Framework | Variation | Version | Read-only mount dự kiến |
| --- | --- | --- | --- | --- | --- |
| TranslateGemma 4B IT | `keras/translategemma` | Keras | `translategemma_4b_it` | `1` | `/kaggle/input/models/keras/translategemma/keras/translategemma_4b_it/1` |
| Whisper Large V3 Turbo | `dangkhoa2016/openai-whisper-large-v3-turbo` | Transformers | `default` | `1` | `/kaggle/input/models/dangkhoa2016/openai-whisper-large-v3-turbo/transformers/default/1` |
| VoxCPM2 | `dangkhoa2016/openbmb-voxcpm2` | PyTorch | `default` | `1` | `/kaggle/input/models/dangkhoa2016/openbmb-voxcpm2/pytorch/default/1` |

Kaggle mount các input đã attach bên dưới `/kaggle/input` dưới dạng **read-only**. Không sửa trực tiếp model file tại đây. Source của repository, virtual environment, log, WAV/ZIP được tạo ra, benchmark output và runtime state phải nằm bên dưới `/kaggle/working`, là vùng có quyền ghi.

Hãy xác nhận cả ba model đã xuất hiện trong panel Kaggle Input trước khi chạy notebook.

## Step 1 — Prepare the public source / Chuẩn bị source công khai

### English

The first code cell clones the public repository into `/kaggle/working` when it is not already present, switches the notebook into that repository, then prints the exact Git `HEAD` and branch status. This makes the source revision visible before any environment setup or model work begins.

**Expected evidence:** the repository path, an exact Git commit SHA, and a clean or explicitly understood branch status.

### Tiếng Việt

Code cell đầu tiên clone repository công khai vào `/kaggle/working` nếu repository chưa tồn tại, chuyển working directory của notebook vào repository, sau đó in chính xác Git `HEAD` và branch status. Nhờ đó source revision được hiển thị rõ trước khi setup environment hoặc chạy model.

**Kết quả cần thấy:** đường dẫn repository, một Git commit SHA chính xác và branch status sạch hoặc có trạng thái đã được hiểu rõ.


In [ ]:
import json
import os
import subprocess
import time
import zipfile
from pathlib import Path

from IPython.display import Audio, display

# Evidence command: git rev-parse HEAD
REPO = Path('/kaggle/working/English-Vietnamese-Speech-API-On-Kaggle-T4x2-GPU')
REPO_URL = 'https://github.com/dangkhoa2016/English-Vietnamese-Speech-API-On-Kaggle-T4x2-GPU.git'
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
print('Repository:', REPO)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
subprocess.run(['git', 'status', '--short', '--branch'], check=True)


## Step 2 — Verify attached Kaggle model inputs / Xác minh các model Kaggle đã attach

### English

This cell fails early if any required Kaggle Model root is missing. The qualified production layout uses Version `1`. Runtime code reads model files directly from the read-only `/kaggle/input` mounts and does not copy model weights into the repository.

**Expected evidence:** three `ATTACHED:` lines, one each for TranslateGemma 4B IT, Whisper Large V3 Turbo, and VoxCPM2.

### Tiếng Việt

Cell này dừng sớm nếu thiếu bất kỳ Kaggle Model root bắt buộc nào. Layout production đã qualification sử dụng Version `1`. Runtime đọc model trực tiếp từ các mount read-only bên dưới `/kaggle/input` và không copy model weight vào repository.

**Kết quả cần thấy:** ba dòng `ATTACHED:`, tương ứng TranslateGemma 4B IT, Whisper Large V3 Turbo và VoxCPM2.


In [ ]:
required_model_roots = {
    'TranslateGemma 4B IT': Path('/kaggle/input/models/keras/translategemma/keras/translategemma_4b_it'),
    'Whisper Large V3 Turbo': Path('/kaggle/input/models/dangkhoa2016/openai-whisper-large-v3-turbo/transformers/default'),
    'VoxCPM2': Path('/kaggle/input/models/dangkhoa2016/openbmb-voxcpm2/pytorch/default'),
}
for name, path in required_model_roots.items():
    assert path.is_dir(), f'Missing attached model: {name}: {path}'
    print(f'ATTACHED: {name}: {path}')


## Step 3 — Bootstrap the qualified dual-environment runtime / Bootstrap runtime hai environment đã qualification

### English

The bootstrap creates the pinned translation/JAX environment and a separate speech/PyTorch environment under writable `/kaggle/working`. Keeping these environments separate preserves the qualified dependency and CUDA boundaries while keeping GPU ownership explicit.

**Expected evidence:** successful completion followed by the qualified JAX, Keras, KerasHub, PyTorch, Transformers, and VoxCPM package versions.

### Tiếng Việt

Bootstrap tạo translation/JAX environment đã pin và speech/PyTorch environment riêng bên dưới `/kaggle/working`, là vùng có quyền ghi. Việc tách hai environment giúp giữ đúng dependency/CUDA boundary đã qualification và làm GPU ownership rõ ràng.

**Kết quả cần thấy:** bootstrap hoàn tất thành công, sau đó in version đã qualification của JAX, Keras, KerasHub, PyTorch, Transformers và VoxCPM.


In [ ]:
subprocess.run(['bash', 'scripts/bootstrap_kaggle.sh'], check=True)
print('Runtime versions:')
subprocess.run(['.venv/bin/python', '-c', "import jax,keras,keras_hub; print('jax',jax.__version__); print('keras',keras.__version__); print('keras_hub',keras_hub.__version__)"], check=True)
subprocess.run(['.venv-speech/bin/python', '-c', "import torch,transformers; import importlib.metadata as m; print('torch',torch.__version__); print('transformers',transformers.__version__); print('voxcpm',m.version('voxcpm'))"], check=True)


## Step 4 — Run hardware and model preflight / Chạy preflight phần cứng và model

### English

Preflight checks the Kaggle T4 x2 topology, required tools, attached model mounts, writable runtime paths, and other fail-closed prerequisites before any real model service starts.

**Expected evidence:** preflight exits with status `0`. A missing accelerator, model mount, or required runtime contract must stop the notebook at this stage.

### Tiếng Việt

Preflight kiểm tra topology Kaggle T4 x2, các tool bắt buộc, model mount đã attach, writable runtime path và những điều kiện fail-closed khác trước khi bất kỳ real-model service nào khởi động.

**Kết quả cần thấy:** preflight exit với status `0`. Nếu thiếu accelerator, model mount hoặc runtime contract bắt buộc thì notebook phải dừng tại bước này.


In [ ]:
env = os.environ.copy()
env['PYTHON_BIN'] = str(REPO / '.venv/bin/python')
subprocess.run(['bash', 'scripts/preflight.sh'], check=True, env=env)


## Step 5 — Run the CPU-testable quality gate / Chạy quality gate có thể kiểm tra trên CPU

### English

Before spending GPU time on model inference, this cell runs Ruff linting, format validation, and the repository pytest suite. These checks validate source and API contracts; they do not replace real-model acceptance on Kaggle T4 x2.

**Expected evidence:** Ruff and pytest pass, followed by `CPU_TEST_GATE=PASS`.

### Tiếng Việt

Trước khi dùng GPU cho model inference, cell này chạy Ruff lint, format validation và toàn bộ pytest suite của repository. Các check này xác minh source/API contract nhưng không thay thế real-model acceptance trên Kaggle T4 x2.

**Kết quả cần thấy:** Ruff và pytest đều PASS, sau đó xuất hiện `CPU_TEST_GATE=PASS`.


In [ ]:
subprocess.run(['.venv/bin/ruff', 'check', 'src', 'tests'], check=True)
subprocess.run(['.venv/bin/ruff', 'format', '--check', 'src', 'tests'], check=True)
subprocess.run(['.venv/bin/python', '-m', 'pytest', '-q'], check=True)
print('CPU_TEST_GATE=PASS')


## Step 6 — Start the real T4 x2 stack with staged cold loading / Khởi động stack T4 x2 thật theo staged cold loading

### English

This cell stops stale services first and verifies a clean GPU-memory baseline. It then starts the real translation and speech backends in the qualified order: translation ready → speech ready → coordinator ready. Parallel cold loading is intentionally avoided because attached-model storage contention was observed during qualification.

**Expected evidence:** `status.sh` reports every service ready, and `nvidia-smi` shows model residency on both Tesla T4 GPUs without OOM.

### Tiếng Việt

Cell này dừng các service cũ trước và xác minh GPU-memory baseline sạch. Sau đó nó khởi động translation backend và speech backend thật theo thứ tự đã qualification: translation ready → speech ready → coordinator ready. Cold loading song song được chủ đích tránh vì qualification từng ghi nhận attached-model storage contention.

**Kết quả cần thấy:** `status.sh` báo tất cả service ready và `nvidia-smi` cho thấy model residency trên cả hai Tesla T4 mà không OOM.


In [ ]:
subprocess.run(['bash', 'scripts/stop.sh'], check=True)
used = subprocess.check_output(['nvidia-smi','--query-gpu=memory.used','--format=csv,noheader,nounits'], text=True)
assert all(int(x.strip()) <= 64 for x in used.splitlines()), f'GPU baseline not clean: {used}'
service_env = os.environ.copy()
service_env.update({
    'SPEECH_BACKEND':'real',
    'TRANSLATION_BACKEND':'real',
    'API_AUTH_REQUIRED': os.getenv('NOTEBOOK_API_AUTH_REQUIRED', 'false'),
    'STARTUP_TIMEOUT_S':'360',
})
subprocess.run(['bash', 'scripts/start.sh'], check=True, env=service_env)
subprocess.run(['bash', 'scripts/status.sh'], check=True)
subprocess.run(['nvidia-smi'], check=True)


## Step 7 — Public transcription demo / Demo public transcription

### English

This cell sends the bundled English WAV fixture only to the public coordinator on `127.0.0.1:8090` and exercises `POST /v1/audio/transcriptions`. The internal speech worker remains private on port `8092`.

**Expected evidence:** a JSON transcription response containing recognized English text and request metadata.

### Tiếng Việt

Cell này gửi WAV tiếng Anh đi kèm chỉ đến public coordinator tại `127.0.0.1:8090` và gọi `POST /v1/audio/transcriptions`. Speech worker nội bộ vẫn private trên port `8092`.

**Kết quả cần thấy:** JSON transcription response chứa English text đã nhận dạng và request metadata.


In [ ]:
demo_dir = REPO / 'runtime/notebook-showcase'
demo_dir.mkdir(parents=True, exist_ok=True)
fixture = REPO / 'data/fixtures/english-short.wav'
transcription_json = demo_dir / 'transcription.json'
with transcription_json.open('wb') as out:
    subprocess.run(['curl','-fsS','-F',f'file=@{fixture};type=audio/wav','http://127.0.0.1:8090/v1/audio/transcriptions'], check=True, stdout=out)
transcription = json.loads(transcription_json.read_text())
print(json.dumps(transcription, ensure_ascii=False, indent=2))


## Step 8 — Public English and Vietnamese TTS demos / Demo public TTS tiếng Anh và tiếng Việt

### English

Both requests use `POST /v1/audio/speech` through coordinator port `8090`. VoxCPM2 remains resident in the speech worker on physical GPU 1. The notebook renders each returned WAV with an inline audio player so a reviewer can listen directly.

**Expected evidence:** two non-empty WAV files, one English and one Vietnamese, plus two playable audio widgets.

### Tiếng Việt

Cả hai request đều gọi `POST /v1/audio/speech` qua coordinator port `8090`. VoxCPM2 tiếp tục resident trong speech worker trên physical GPU 1. Notebook hiển thị audio player cho từng WAV trả về để reviewer có thể nghe trực tiếp.

**Kết quả cần thấy:** hai WAV không rỗng, một tiếng Anh và một tiếng Việt, cùng hai audio widget có thể phát.


In [ ]:
tts_cases = {
    'en': 'Hello from the Kaggle T4x2 production showcase.',
    'vi': 'Xin chào từ bản trình diễn production trên Kaggle T4x2.',
}
for language, text in tts_cases.items():
    text_path = demo_dir / f'tts-{language}.txt'
    wav_path = demo_dir / f'tts-{language}.wav'
    text_path.write_text(text, encoding='utf-8')
    subprocess.run(['curl','-fsS','-F',f'file=@{text_path};type=text/plain','-F',f'language={language}','http://127.0.0.1:8090/v1/audio/speech','-o',str(wav_path)], check=True)
    print(language, wav_path, wav_path.stat().st_size, 'bytes')
    display(Audio(filename=str(wav_path)))


## Step 9 — Flagship English audio → Vietnamese speech pipeline / Pipeline chính từ audio tiếng Anh → giọng nói tiếng Việt

### English

This is the end-to-end production showcase: normalize input → Whisper ASR → TranslateGemma translation → VoxCPM2 TTS → ZIP. The cell validates the exact four-member ZIP contract before printing the transcript, translation, metadata, and Vietnamese audio.

**Expected evidence:** the ZIP contains exactly `transcript.en.txt`, `translation.vi.txt`, `speech.vi.wav`, and `metadata.json`.

### Tiếng Việt

Đây là production showcase end-to-end: chuẩn hóa input → Whisper ASR → TranslateGemma translation → VoxCPM2 TTS → ZIP. Cell xác minh chính xác ZIP contract gồm bốn file trước khi in transcript, translation, metadata và phát audio tiếng Việt.

**Kết quả cần thấy:** ZIP chứa chính xác `transcript.en.txt`, `translation.vi.txt`, `speech.vi.wav` và `metadata.json`.


In [ ]:
flagship_zip = demo_dir / 'flagship.zip'
subprocess.run(['curl','-fsS','-F',f'file=@{fixture};type=audio/wav','http://127.0.0.1:8090/v1/audio/translate-and-speak','-o',str(flagship_zip)], check=True)
extract_dir = demo_dir / 'flagship'
extract_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(flagship_zip) as archive:
    names = set(archive.namelist())
    expected = {'transcript.en.txt','translation.vi.txt','speech.vi.wav','metadata.json'}
    assert names == expected, names
    archive.extractall(extract_dir)
transcript = (extract_dir/'transcript.en.txt').read_text(encoding='utf-8').strip()
translation = (extract_dir/'translation.vi.txt').read_text(encoding='utf-8').strip()
metadata = json.loads((extract_dir/'metadata.json').read_text(encoding='utf-8'))
print('EN:', transcript)
print('VI:', translation)
print(json.dumps(metadata, ensure_ascii=False, indent=2))
display(Audio(filename=str(extract_dir/'speech.vi.wav')))


## Step 10 — Production Scenario Gallery / Bộ tình huống production thực tế

### English

This gallery demonstrates coordinated multi-model behavior instead of isolated endpoint checks. For each scenario, VoxCPM2 first synthesizes a realistic English source utterance through the public speech endpoint. That generated WAV is then sent back through the public flagship endpoint, exercising Whisper ASR → TranslateGemma EN→VI translation → VoxCPM2 Vietnamese TTS as one production flow. The original bundled WAV fixture remains the independent external/static case from the previous step.

The four scenarios cover customer support, a technical incident, travel and hospitality, and a business update. Each result keeps the exact four-file ZIP contract, prints the request ID, English source-TTS latency, and pipeline stage timings, and renders both the generated English source audio and the final Vietnamese audio for direct review.

**Expected evidence:** four completed scenario rows, four distinct request IDs, non-empty transcript/translation artifacts, playable source and destination audio, and `SCENARIO_GALLERY=PASS`.

### Tiếng Việt

Gallery này trình diễn khả năng phối hợp nhiều model thay vì chỉ kiểm tra từng endpoint riêng lẻ. Với mỗi tình huống, VoxCPM2 trước tiên tạo một đoạn giọng nói tiếng Anh thực tế thông qua public speech endpoint. WAV vừa tạo sau đó được gửi lại qua public flagship endpoint để chạy toàn bộ chuỗi Whisper ASR → TranslateGemma dịch EN→VI → VoxCPM2 TTS tiếng Việt như một production flow thống nhất. WAV fixture đi kèm ở bước trước vẫn được giữ làm case external/static độc lập.

Bốn tình huống bao gồm hỗ trợ khách hàng, sự cố kỹ thuật, du lịch và khách sạn, cùng cập nhật công việc. Mỗi kết quả vẫn giữ chính xác ZIP contract bốn file, in request ID, latency của English source TTS và thời gian từng stage trong pipeline, đồng thời hiển thị cả audio nguồn tiếng Anh và audio tiếng Việt cuối để reviewer nghe trực tiếp.

**Evidence mong đợi:** bốn scenario hoàn tất, bốn request ID khác nhau, transcript/translation không rỗng, source/destination audio phát được và marker `SCENARIO_GALLERY=PASS`.


In [ ]:
scenario_cases = [
    {
        'slug': 'customer-support',
        'title': 'Customer Support',
        'text': (
            'Hello, I am calling because my order was marked as delivered yesterday, '
            'but the package never arrived. I checked with the front desk and my neighbors. '
            'Please help me verify the delivery status and arrange a replacement if necessary.'
        ),
    },
    {
        'slug': 'technical-incident',
        'title': 'Technical Incident',
        'text': (
            'We are investigating elevated API latency in the production environment. '
            'The database is healthy, but requests through the payment service are timing out. '
            'Traffic has been reduced while the team validates the recovery plan.'
        ),
    },
    {
        'slug': 'travel-hospitality',
        'title': 'Travel and Hospitality',
        'text': (
            'I have a reservation for two guests next Friday and expect to arrive around nine in the evening. '
            'Could you confirm late check-in, arrange airport pickup, and let me know whether breakfast is included?'
        ),
    },
    {
        'slug': 'business-update',
        'title': 'Business Update',
        'text': (
            'This week the team completed the API migration and reduced average response time by eighteen percent. '
            'The remaining task is production monitoring, and we plan to complete the rollout after the final review tomorrow.'
        ),
    },
]

scenario_root = demo_dir / 'production-scenarios'
scenario_root.mkdir(parents=True, exist_ok=True)
expected_members = {'transcript.en.txt', 'translation.vi.txt', 'speech.vi.wav', 'metadata.json'}
scenario_summary = []

for case in scenario_cases:
    slug = case['slug']
    case_dir = scenario_root / slug
    case_dir.mkdir(parents=True, exist_ok=True)
    source_text = case_dir / 'source.en.txt'
    source_wav = case_dir / 'source.en.wav'
    source_text.write_text(case['text'], encoding='utf-8')

    source_tts_started = time.perf_counter()
    subprocess.run([
        'curl', '-fsS',
        '-F', f'file=@{source_text};type=text/plain',
        '-F', 'language=en',
        'http://127.0.0.1:8090/v1/audio/speech',
        '-o', str(source_wav),
    ], check=True)
    source_tts_ms = round((time.perf_counter() - source_tts_started) * 1000, 3)
    assert source_wav.stat().st_size > 44, f'{slug}: empty English source WAV'

    archive_path = case_dir / 'result.zip'
    subprocess.run([
        'curl', '-fsS',
        '-F', f'file=@{source_wav};type=audio/wav',
        'http://127.0.0.1:8090/v1/audio/translate-and-speak',
        '-o', str(archive_path),
    ], check=True)

    result_dir = case_dir / 'result'
    result_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        names = set(archive.namelist())
        assert names == expected_members, f'{slug}: unexpected ZIP members: {sorted(names)}'
        archive.extractall(result_dir)

    transcript = (result_dir / 'transcript.en.txt').read_text(encoding='utf-8').strip()
    translation = (result_dir / 'translation.vi.txt').read_text(encoding='utf-8').strip()
    metadata = json.loads((result_dir / 'metadata.json').read_text(encoding='utf-8'))
    vi_wav = result_dir / 'speech.vi.wav'
    assert transcript, f'{slug}: empty transcript'
    assert translation, f'{slug}: empty translation'
    assert vi_wav.stat().st_size > 44, f'{slug}: empty Vietnamese WAV'

    row = {
        'scenario': case['title'],
        'request_id': metadata['request_id'],
        'source_tts_ms': source_tts_ms,
        'asr_ms': metadata['timings_ms']['asr'],
        'translation_ms': metadata['timings_ms']['translation'],
        'tts_ms': metadata['timings_ms']['tts'],
        'total_ms': metadata['timings_ms']['total'],
        'transcript': transcript,
        'translation': translation,
    }
    scenario_summary.append(row)

    print(f"\n=== {case['title']} ===")
    print('Original EN:', case['text'])
    print('Whisper EN:', transcript)
    print('TranslateGemma VI:', translation)
    print('Request ID:', metadata['request_id'])
    print('English source TTS (ms):', source_tts_ms)
    print('Pipeline timings (ms):', json.dumps(metadata['timings_ms'], ensure_ascii=False))
    print('English source audio:')
    display(Audio(filename=str(source_wav)))
    print('Vietnamese output audio:')
    display(Audio(filename=str(vi_wav)))

assert len(scenario_summary) == 4
assert len({row['request_id'] for row in scenario_summary}) == 4

print('\n=== Production Scenario Summary ===')
for row in scenario_summary:
    print(
        f"{row['scenario']}: request_id={row['request_id']} | "
        f"Source TTS={row['source_tts_ms']} ms | ASR={row['asr_ms']} ms | "
        f"Translation={row['translation_ms']} ms | "
        f"TTS={row['tts_ms']} ms | Total={row['total_ms']} ms"
    )
print(json.dumps(scenario_summary, ensure_ascii=False, indent=2))
print('SCENARIO_GALLERY=PASS')


## Step 11 — Run a bounded warm benchmark / Chạy bounded warm benchmark

### English

The full release qualification used a larger retained benchmark. The public template intentionally runs one sequential warm request and disables the two-request stress test so a fresh showcase remains bounded while still exposing stage timing and standalone measurements.

**Expected evidence:** a benchmark JSON summary containing sequential pipeline timing and standalone measurements.

### Tiếng Việt

Release qualification đầy đủ đã dùng benchmark lớn hơn. Public template chủ đích chỉ chạy một sequential warm request và tắt stress test hai request để fresh showcase có giới hạn hợp lý nhưng vẫn hiển thị stage timing và các standalone measurement.

**Kết quả cần thấy:** benchmark JSON summary chứa sequential pipeline timing và standalone measurement.


In [ ]:
bench_env = os.environ.copy()
bench_env.update({'SEQUENTIAL_RUNS':'1','RUN_CONCURRENT_QUALIFICATION':'false'})
subprocess.run(['bash', 'scripts/benchmark_pipeline.sh'], check=True, env=bench_env)
bench = json.loads((REPO/'runtime/benchmark-pipeline/results.json').read_text())
print(json.dumps({'sequential_summary':bench['sequential_summary'],'standalone':bench['standalone']}, ensure_ascii=False, indent=2))


## Step 12 — Optional authenticated Cloudflare Quick Tunnel / Cloudflare Quick Tunnel có authentication, tùy chọn

### English

Public exposure is disabled by default. A tunnel may start only when both `ENABLE_CLOUDFLARE_TUNNEL=true` and `NOTEBOOK_API_AUTH_REQUIRED=true`. The script exposes only coordinator port `8090`. The bearer key remains in `runtime/api_key.txt` and is never printed by this notebook.

**Expected evidence:** the default run prints `Cloudflare tunnel disabled (default).`. An authenticated opt-in run may start the tunnel without exposing the API key.

### Tiếng Việt

Public exposure mặc định bị tắt. Tunnel chỉ được phép chạy khi cả `ENABLE_CLOUDFLARE_TUNNEL=true` và `NOTEBOOK_API_AUTH_REQUIRED=true`. Script chỉ expose coordinator port `8090`. Bearer key vẫn nằm trong `runtime/api_key.txt` và notebook không in giá trị key.

**Kết quả cần thấy:** run mặc định in `Cloudflare tunnel disabled (default).`. Run opt-in có authentication có thể khởi động tunnel nhưng không được làm lộ API key.


In [ ]:
tunnel_proc = None
if os.getenv('ENABLE_CLOUDFLARE_TUNNEL', 'false').lower() == 'true':
    assert os.getenv('NOTEBOOK_API_AUTH_REQUIRED', 'false').lower() == 'true', 'Refusing tunnel without API auth'
    tunnel_env = os.environ.copy()
    tunnel_env.update({'ENABLE_CLOUDFLARE_TUNNEL':'true','API_AUTH_REQUIRED':'true'})
    tunnel_proc = subprocess.Popen(['bash','scripts/run_tunnel.sh'], env=tunnel_env)
    time.sleep(3)
    print('Tunnel process started. API key is stored in runtime/api_key.txt and is not printed by this notebook.')
else:
    print('Cloudflare tunnel disabled (default).')


## Step 13 — Clean stop and final evidence / Dừng sạch và in evidence cuối

### English

The final cell terminates the optional tunnel when present, stops the coordinator and workers, waits briefly for teardown, prints GPU memory usage, verifies that ports `8090`, `8091`, and `8092` are released, and emits the final showcase marker.

**Expected evidence:** `PORT_CLEANUP=PASS` followed by `KAGGLE_PRODUCTION_SHOWCASE=PASS`.

### Tiếng Việt

Cell cuối dừng tunnel nếu có, stop coordinator và các worker, chờ teardown trong thời gian ngắn, in GPU memory usage, xác minh các port `8090`, `8091` và `8092` đã được giải phóng, sau đó xuất final showcase marker.

**Kết quả cần thấy:** `PORT_CLEANUP=PASS` rồi `KAGGLE_PRODUCTION_SHOWCASE=PASS`.


In [ ]:
if tunnel_proc is not None and tunnel_proc.poll() is None:
    tunnel_proc.terminate()
    tunnel_proc.wait(timeout=10)
subprocess.run(['bash','scripts/stop.sh'], check=True)
time.sleep(2)
subprocess.run(['nvidia-smi','--query-gpu=index,name,memory.used','--format=csv,noheader,nounits'], check=True)
subprocess.run(['.venv/bin/python','-c',"from en_vi_speech_api.coordinator.lifecycle import ports_are_free; assert ports_are_free((8090,8091,8092)); print('PORT_CLEANUP=PASS')"], check=True)
print('KAGGLE_PRODUCTION_SHOWCASE=PASS')
